# 🏛️ VoxCity 랜드마크 가시성 분석

3D 레이 캐스팅(Ray-casting)을 사용하여 연구 구역 내의 모든 위치에서 특정 랜드마크(건물)의 **가시성**을 분석합니다.

## 활용 사례

- **길 찾기 연구** - 여러 위치에서 어떤 랜드마크가 보이는가?
- **도시 계획** - 신축 건물의 시각적 영향 평가
- **부동산** - 다양한 매물에서의 조망권 평가
- **관광** - 주요 명소의 가시성 매핑

## 랜드마크 선택 방법

| 방법 | 사용 시점 |
|--------|-------------|
| `landmark_building_ids` | 특정 건물의 ID를 알고 있을 때 |
| `landmark_polygon` | 랜드마크를 포함하는 사용자 정의 영역을 설정할 때 |
| 자동 선택 (중심부) | VoxCity가 구역 중심 근처의 건물을 자동으로 선택하게 할 때 |

## 사전 요구 사항

```python
pip install voxcity
```

In [ ]:
# %pip install voxcity

from voxcity.generator import get_voxcity
from voxcity.geoprocessor.draw import draw_rectangle_map_cityname
from voxcity.simulator.view import get_landmark_visibility_map

meshsize = 5
cityname = "Tokyo, Japan"

rectangle_vertices = [
    (139.760, 35.680),  # 남서(SW)
    (139.760, 35.690),  # 북서(NW)
    (139.770, 35.690),  # 북동(NE)
    (139.770, 35.680)   # 남동(SE)
]

city = get_voxcity(
    rectangle_vertices,
    meshsize=meshsize,
    building_source='OpenStreetMap',
    land_cover_source='OpenStreetMap',
    canopy_height_source='High Resolution 1m Global Canopy Height Maps',
    dem_source='DeltaDTM',
    output_dir='output/landmark_demo'
)

# VoxCity 객체에서 building_gdf에 접근
building_gdf = city.extras.get('building_gdf', None)
len(building_gdf) if building_gdf is not None else 0


---
## 🎯 랜드마크 선택

어떤 건물을 "랜드마크"로 지정할 것인지에 대한 세 가지 방법:

1. **ID 기준**: `landmark_building_ids` 제공 - GeoDataFrame의 건물 인덱스 리스트
2. **폴리곤 기준**: `landmark_polygon` 제공 - GeoJSON 스타일의 폴리곤 영역 내부의 모든 건물이 랜드마크가 됨
3. **자동 중심부**: 두 매개변수를 생략 - 구역 사각형의 중심에 있는 건물을 자동으로 선택

In [ ]:
landmark_kwargs = {
    "view_point_height": 1.5,
    "colormap": "cool",
    "obj_export": True,
    "output_directory": "output/landmark_demo",
    "output_file_name": "landmark_visibility",
    "alpha": 1.0,
}

# 예시 1: ID로 선택 (상위 몇 개를 샘플로 선택)
ids_sample = building_gdf.head(3).index.tolist() if building_gdf is not None else []

landmark_vis_map, voxcity_grid_marked = get_landmark_visibility_map(
    city,
    building_gdf=building_gdf,
    landmark_building_ids=ids_sample,
    **landmark_kwargs
)

landmark_vis_map.shape


In [ ]:
# 예시 2: 구역 중심점을 사용하여 자동 선택 (ID와 폴리곤 생략)
landmark_vis_map_auto, _ = get_landmark_visibility_map(
    city,
    building_gdf=building_gdf,
    rectangle_vertices=rectangle_vertices,
    **landmark_kwargs
)

landmark_vis_map_auto.shape


---
## 📊 결과 해석

- **가시성 값**: 0 = 보이지 않음, 1 = 완전히 보임
- **출력**: 각 위치에서의 가시성을 보여주는 색상화된 OBJ 메쉬
- **`voxcity_grid_marked`**: 랜드마크가 강조된 3D 복셀 그리드

## 다음 단계

- `demo_view.ipynb` - Green View Index & Sky View Index
- `demo_solar.ipynb` - Solar irradiance simulation
- `demo_3d_visualization.ipynb` - Advanced 3D visualization